## Exploración de la población estudiantil universitaria del Perú
### Censos universitarios de 1996 y 2010 y registros TUNI 2020–2025

In [49]:
#%pip install pyreadstat

In [1]:
# Paquetes
from pathlib import Path

import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

#### **Censo 1996**

In [ ]:
# Ruta de la fuente
ruta_1996 = Path(r"E:\BD\Censo Universitario\1996")

# Carpetas de estudiantes
carpetas_1996 = {
    "Pregrado": ruta_1996 / "1 Estudiante de Pre-Grado",
    "Posgrado": ruta_1996 / "2 Estudiante de Post-Grado",
}

# Inventario de archivos SPSS
archivos_1996 = []

for nivel, carpeta in carpetas_1996.items():
    for archivo in sorted(carpeta.glob("*.sav")):
        archivos_1996.append({
            "nivel": nivel,
            "archivo": archivo.name,
            "tamano_mb": round(archivo.stat().st_size / 1024**2, 2),
            "ruta": archivo,
        })

inventario_1996 = pd.DataFrame(archivos_1996)

display(inventario_1996[["nivel", "archivo", "tamano_mb"]])

,nivel,archivo,tamano_mb
0,Pregrado,M1S01.sav,44.50
1,Pregrado,M1S02.sav,59.29
2,Pregrado,M1S03.sav,40.91
3,Pregrado,M1S04.sav,55.93
4,Pregrado,M1S05.sav,38.49
5,Posgrado,M2S01.sav,1.32
6,Posgrado,M2S02.sav,1.16
7,Posgrado,M2S03.sav,1.59
8,Posgrado,M2S04.sav,0.21
9,Posgrado,M2S05.sav,1.97


In [3]:
# Revisar los metadatos de cada archivo
resumen_1996 = []

for fila in inventario_1996.itertuples(index=False):
    _, meta = pyreadstat.read_sav(
        str(fila.ruta),
        metadataonly=True,
    )

    resumen_1996.append({
        "nivel": fila.nivel,
        "archivo": fila.archivo,
        "registros": meta.number_rows,
        "variables": meta.number_columns,
        "contiene_key": any(
            nombre.strip().upper() == "KEY"
            for nombre in meta.column_names
        ),
    })

resumen_1996 = pd.DataFrame(resumen_1996)

display(resumen_1996)

,nivel,archivo,registros,variables,contiene_key
0,Pregrado,M1S01.sav,339969,13,True
1,Pregrado,M1S02.sav,335714,105,True
2,Pregrado,M1S03.sav,335714,61,True
3,Pregrado,M1S04.sav,335714,105,True
4,Pregrado,M1S05.sav,335714,42,True
5,Posgrado,M2S01.sav,10818,14,True
6,Posgrado,M2S02.sav,10818,71,True
7,Posgrado,M2S03.sav,22021,10,True
8,Posgrado,M2S04.sav,2762,12,True
9,Posgrado,M2S05.sav,10818,98,True


In [4]:
# Conteos iniciales de estudiantes: Censo 1996
archivos_identificacion_1996 = {
    "Pregrado": carpetas_1996["Pregrado"] / "M1S01.sav",
    "Posgrado": carpetas_1996["Posgrado"] / "M2S01.sav",
}

conteos_1996 = []

for nivel, archivo in archivos_identificacion_1996.items():
    # Leer únicamente el identificador
    datos, _ = pyreadstat.read_sav(
        str(archivo),
        usecols=["key"],
    )

    # Limpiar espacios y reconocer claves vacías
    claves = datos["key"].astype("string").str.strip()
    claves = claves.mask(claves == "")

    registros = len(datos)
    sin_clave = int(claves.isna().sum())
    claves_unicas = int(claves.nunique())

    conteos_1996.append({
        "nivel": nivel,
        "archivo": archivo.name,
        "registros": registros,
        "claves_unicas": claves_unicas,
        "registros_sin_clave": sin_clave,
        "repeticiones_adicionales": registros - sin_clave - claves_unicas,
    })

conteos_1996 = pd.DataFrame(conteos_1996)

display(conteos_1996)

,nivel,archivo,registros,claves_unicas,registros_sin_clave,repeticiones_adicionales
0,Pregrado,M1S01.sav,339969,339955,0,14
1,Posgrado,M2S01.sav,10818,10818,0,0


In [5]:
# Pregrado 1996
pregrado_1996, meta_pregrado_1996 = pyreadstat.read_sav(str(carpetas_1996["Pregrado"] / "M1S01.sav"))

In [30]:
tabla_variables_1996_pre = pd.DataFrame({
    "Descripción": meta_pregrado_1996.column_names_to_labels,
    "Tipo": pregrado_1996.dtypes.astype(str),
    "Escala SPSS": meta_pregrado_1996.variable_measure,
    "Etiquetas": meta_pregrado_1996.variable_value_labels,
}, index=pregrado_1996.columns).rename_axis("Variable")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabla_variables_1996_pre)

,Descripción,Tipo,Escala SPSS,Etiquetas
Variable,,,,
key,None,str,nominal,NaN
seccion,None,str,nominal,NaN
p3,None,float64,scale,NaN
p4,None,float64,scale,NaN
po,None,float64,scale,NaN
p1_4,None,str,nominal,NaN
p1_4a,None,str,nominal,NaN
p2,None,str,nominal,NaN
p2_nomdpto,None,str,nominal,NaN


In [7]:
# Posgrado 1996
posgrado_1996, meta_posgrado_1996 = pyreadstat.read_sav(str(carpetas_1996["Posgrado"] / "M2S01.sav"))

In [31]:
tabla_variables_1996_pos = pd.DataFrame({
    "Descripción": meta_pregrado_1996.column_names_to_labels,
    "Tipo": pregrado_1996.dtypes.astype(str),
    "Escala SPSS": meta_pregrado_1996.variable_measure,
    "Etiquetas": meta_pregrado_1996.variable_value_labels,
}, index=pregrado_1996.columns).rename_axis("Variable")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabla_variables_1996_pos)

,Descripción,Tipo,Escala SPSS,Etiquetas
Variable,,,,
key,None,str,nominal,NaN
seccion,None,str,nominal,NaN
p3,None,float64,scale,NaN
p4,None,float64,scale,NaN
po,None,float64,scale,NaN
p1_4,None,str,nominal,NaN
p1_4a,None,str,nominal,NaN
p2,None,str,nominal,NaN
p2_nomdpto,None,str,nominal,NaN


In [ ]:
# Registros y porcentajes por departamento: Censo 1996
def contar_por_departamento(datos):
    departamento = (
        datos["p2_nomdpto"].astype("string").str.strip().str.upper().replace("", pd.NA).fillna("SIN INFORMACIÓN")
    )

    return departamento.value_counts()


tabla_departamentos_1996 = pd.concat(
    [
        contar_por_departamento(pregrado_1996).rename("Pregrado"),
        contar_por_departamento(posgrado_1996).rename("Posgrado"),
    ],
    axis=1,
).fillna(0).astype(int).sort_index()

tabla_departamentos_1996.index.name = "Departamento"

# Comprobar los totales
assert tabla_departamentos_1996["Pregrado"].sum() == len(pregrado_1996)
assert tabla_departamentos_1996["Posgrado"].sum() == len(posgrado_1996)

# Porcentajes dentro de cada nivel
for nivel in ["Pregrado", "Posgrado"]:
    tabla_departamentos_1996[f"{nivel} (%)"] = (tabla_departamentos_1996[nivel] / tabla_departamentos_1996[nivel].sum() * 100)

# Ordenar las columnas
tabla_departamentos_1996 = tabla_departamentos_1996[["Pregrado", "Pregrado (%)", "Posgrado", "Posgrado (%)"]]

# Agregar fila total
tabla_departamentos_1996.loc["TOTAL"] = tabla_departamentos_1996.sum()

display(
    tabla_departamentos_1996.style.format({
        "Pregrado": "{:,.0f}",
        "Pregrado (%)": "{:.2f} %",
        "Posgrado": "{:,.0f}",
        "Posgrado (%)": "{:.2f} %",
    })
)

,Pregrado,Pregrado (%),Posgrado,Posgrado (%)
Departamento,,,,
AMAZONAS,102,0.03 %,4,0.04 %
ANCASH,"9,351",2.75 %,89,0.82 %
APURIMAC,"2,062",0.61 %,3,0.03 %
AREQUIPA,"26,931",7.92 %,384,3.55 %
AYACUCHO,"7,044",2.07 %,1,0.01 %
CAJAMARCA,"4,676",1.38 %,109,1.01 %
CALLAO,"10,023",2.95 %,370,3.42 %
CUSCO,"13,368",3.93 %,171,1.58 %
HUANCAVELICA,"1,750",0.51 %,3,0.03 %


In [14]:
# Población de posgrado por nivel: Censo 1996
datos_nivel_1996 = posgrado_1996.assign(
    Nivel_academico=posgrado_1996["p4_a"].map({
        1: "Segunda especialidad",
        2: "Maestría",
        3: "Doctorado",
    }).fillna("SIN INFORMACIÓN / OTRO"),
    Identificador=(
        posgrado_1996["key"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    ),
)

tabla_nivel_detallado_1996 = (
    datos_nivel_1996
    .groupby("Nivel_academico", sort=False)
    .agg(
        Registros_totales=("Identificador", "size"),
        Personas_unicas=("Identificador", "nunique"),
        Registros_sin_identificador=(
            "Identificador", lambda x: x.isna().sum()
        ),
    )
)

tabla_nivel_detallado_1996.index.name = "Nivel académico"

tabla_nivel_detallado_1996.loc["TOTAL"] = [
    len(datos_nivel_1996),
    datos_nivel_1996["Identificador"].nunique(),
    datos_nivel_1996["Identificador"].isna().sum(),
]

display(tabla_nivel_detallado_1996.style.format("{:,.0f}"))

,Registros_totales,Personas_unicas,Registros_sin_identificador
Nivel académico,,,
Maestría,"7,921","7,921",0
Segunda especialidad,"2,597","2,597",0
Doctorado,300,300,0
TOTAL,"10,818","10,818",0


#### **Censo 2010**

In [20]:
# Rutas
ruta_2010 = Path(r"E:\BD\Censo Universitario\2010")

archivos_2010 = {
    "Pregrado": ruta_2010 / "01_CENAUN_CED01_PRE-GRADO.sav",
    "Posgrado": ruta_2010 / "02_CENAUN_CED02_POST-GRADO.sav",
}

# Pregrado
pregrado_2010, meta_pregrado_2010 = pyreadstat.read_sav(str(archivos_2010["Pregrado"]))

# Posgrado
posgrado_2010, meta_posgrado_2010 = pyreadstat.read_sav(str(archivos_2010["Posgrado"]))

In [21]:
# Conteos iniciales: Censo 2010
bases_2010 = {"Pregrado": pregrado_2010, "Posgrado": posgrado_2010,}

conteos_2010 = []

for nivel, datos in bases_2010.items():
    identificadores = (
        datos["cap1_IDX_PERSONA"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    registros = len(datos)
    sin_identificador = int(identificadores.isna().sum())
    identificadores_unicos = int(identificadores.nunique())

    conteos_2010.append({
        "nivel": nivel,
        "registros": registros,
        "identificadores_unicos": identificadores_unicos,
        "registros_sin_identificador": sin_identificador,
        "repeticiones_adicionales": (
            registros - sin_identificador - identificadores_unicos
        ),
    })

conteos_2010 = pd.DataFrame(conteos_2010)

display(conteos_2010)

,nivel,registros,identificadores_unicos,registros_sin_identificador,repeticiones_adicionales
0,Pregrado,782970,782846,0,124
1,Posgrado,56358,56046,0,312


In [28]:
tabla_variables_2010_pre = pd.DataFrame({
    "Descripción": meta_pregrado_2010.column_names_to_labels,
    "Tipo": pregrado_2010.dtypes.astype(str),
    "Escala SPSS": meta_pregrado_2010.variable_measure,
    "Etiquetas": meta_pregrado_2010.variable_value_labels,
}, index=pregrado_2010.columns).rename_axis("Variable")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabla_variables_2010_pre)

,Descripción,Tipo,Escala SPSS,Etiquetas
Variable,,,,
cap1_IDX_PERSONA,NaN,str,nominal,NaN
cap1_IDX_MARCO,NaN,str,nominal,NaN
cap1_FECREG,NaN,datetime64[us],scale,NaN
CCPAIS,CÓDIGO DE PAIS -RESIDENCIA,str,nominal,NaN
cap1_PAIS_NOMBRE,NOMBRE DE PAIS - RESIDENCIA,str,nominal,NaN
CCDD,CÓDIGO DEPARTAMENTO - RESIDENCIA,str,nominal,NaN
cap1_DPTO_NOMBRE,NOMBRE DE DEPARTAMENTO -RESIDENCIA,str,nominal,NaN
CCPP,CÓDIGO PROVINCIA -RESIDENCIA,str,nominal,NaN
cap1_PROV_NOMBRE,NOMBRE DE PROVINCIA - RESIDENCIA,str,nominal,NaN


In [29]:
tabla_variables_2010_pos = pd.DataFrame({
    "Descripción": meta_pregrado_2010.column_names_to_labels,
    "Tipo": pregrado_2010.dtypes.astype(str),
    "Escala SPSS": meta_pregrado_2010.variable_measure,
    "Etiquetas": meta_pregrado_2010.variable_value_labels,
}, index=pregrado_2010.columns).rename_axis("Variable")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabla_variables_2010_pos)

,Descripción,Tipo,Escala SPSS,Etiquetas
Variable,,,,
cap1_IDX_PERSONA,NaN,str,nominal,NaN
cap1_IDX_MARCO,NaN,str,nominal,NaN
cap1_FECREG,NaN,datetime64[us],scale,NaN
CCPAIS,CÓDIGO DE PAIS -RESIDENCIA,str,nominal,NaN
cap1_PAIS_NOMBRE,NOMBRE DE PAIS - RESIDENCIA,str,nominal,NaN
CCDD,CÓDIGO DEPARTAMENTO - RESIDENCIA,str,nominal,NaN
cap1_DPTO_NOMBRE,NOMBRE DE DEPARTAMENTO -RESIDENCIA,str,nominal,NaN
CCPP,CÓDIGO PROVINCIA -RESIDENCIA,str,nominal,NaN
cap1_PROV_NOMBRE,NOMBRE DE PROVINCIA - RESIDENCIA,str,nominal,NaN


In [32]:
# Registros y porcentajes por departamento: Censo 2010
def contar_por_departamento_2010(datos):
    departamento = (
        datos["cap1_DPTO_NOMBRE"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
        .fillna("SIN INFORMACIÓN")
    )

    return departamento.value_counts()


tabla_departamentos_2010 = pd.concat(
    [
        contar_por_departamento_2010(pregrado_2010).rename("Pregrado"),
        contar_por_departamento_2010(posgrado_2010).rename("Posgrado"),
    ],
    axis=1,
).fillna(0).astype(int).sort_index()

tabla_departamentos_2010.index.name = "Departamento"

# Comprobar que todos los registros estén incluidos
assert tabla_departamentos_2010["Pregrado"].sum() == len(pregrado_2010)
assert tabla_departamentos_2010["Posgrado"].sum() == len(posgrado_2010)

# Porcentajes dentro de cada nivel
for nivel in ["Pregrado", "Posgrado"]:
    tabla_departamentos_2010[f"{nivel} (%)"] = (
        tabla_departamentos_2010[nivel]
        / tabla_departamentos_2010[nivel].sum()
        * 100
    )

# Ordenar columnas
tabla_departamentos_2010 = tabla_departamentos_2010[
    ["Pregrado", "Pregrado (%)", "Posgrado", "Posgrado (%)"]
]

# Agregar totales
tabla_departamentos_2010.loc["TOTAL"] = (
    tabla_departamentos_2010.sum()
)

display(
    tabla_departamentos_2010.style.format({
        "Pregrado": "{:,.0f}",
        "Pregrado (%)": "{:.2f} %",
        "Posgrado": "{:,.0f}",
        "Posgrado (%)": "{:.2f} %",
    })
)

,Pregrado,Pregrado (%),Posgrado,Posgrado (%)
Departamento,,,,
AMAZONAS,"2,851",0.36 %,281,0.50 %
ANCASH,"31,924",4.08 %,"1,659",2.94 %
APURIMAC,"11,004",1.41 %,652,1.16 %
AREQUIPA,"53,977",6.89 %,"3,634",6.45 %
AYACUCHO,"14,358",1.83 %,"1,417",2.51 %
CAJAMARCA,"18,843",2.41 %,"1,349",2.39 %
CUSCO,"31,507",4.02 %,"2,195",3.89 %
HUANCAVELICA,"6,427",0.82 %,576,1.02 %
HUANUCO,"19,197",2.45 %,"1,854",3.29 %


In [34]:
# Población de posgrado por nivel: Censo 2010
datos_nivel_2010 = posgrado_2010.rename(columns=str.lower).copy()
datos_nivel_2010["Nivel_academico"] = "No especifica"

for variable, nivel in [
    ("codcarrprof_2esp", "Segunda especialidad"),
    ("codcarrprof_maes", "Maestría"),
    ("codcarrprof_doct", "Doctorado"),
]:
    tiene_dato = (
        datos_nivel_2010[variable]
        .astype("string")
        .str.strip()
        .fillna("")
        .ne("")
    )
    datos_nivel_2010.loc[tiene_dato, "Nivel_academico"] = nivel

datos_nivel_2010["Identificador"] = (
    datos_nivel_2010["cap1_idx_persona"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

tabla_nivel_detallado_2010 = (
    datos_nivel_2010
    .groupby("Nivel_academico", sort=False)
    .agg(
        Registros_totales=("Identificador", "size"),
        Personas_unicas=("Identificador", "nunique"),
        Registros_sin_identificador=(
            "Identificador", lambda x: x.isna().sum()
        ),
    )
)

tabla_nivel_detallado_2010.index.name = "Nivel académico"

tabla_nivel_detallado_2010.loc["TOTAL"] = [
    len(datos_nivel_2010),
    datos_nivel_2010["Identificador"].nunique(),
    datos_nivel_2010["Identificador"].isna().sum(),
]

display(tabla_nivel_detallado_2010.style.format("{:,.0f}"))

,Registros_totales,Personas_unicas,Registros_sin_identificador
Nivel académico,,,
Maestría,"44,577","44,470",0
Doctorado,"4,047","4,033",0
Segunda especialidad,"7,726","7,707",0
No especifica,8,8,0
TOTAL,"56,358","56,046",0


#### **TUNI: matrícula 2025-I**

In [36]:
ruta_tuni = Path(r"E:\BD\SUNEDU\TUNI\corte_16092026\3_matriculados")
archivo_tuni_2025_I = ruta_tuni / "matriculado_2025_I.txt"

# Columnas necesarias para esta exploración
columnas_tuni = [
    "CODIGO_INEI",
    "NOMBRE_ENTIDAD",
    "TIPO_ENTIDAD",
    "TIPO_GESTION",
    "PERIODO",
    "PERIODO_ESTANDARIZADO",
    "NIVEL_ACADEMICO",
    "DEPARTAMENTO_LOCAL",
    "GUID_PERSONA",
]

# Cargar datos
tuni_2025_I = pd.read_csv(
    archivo_tuni_2025_I,
    sep="|",
    encoding="cp1252",
    dtype="string",
    usecols=columnas_tuni,
    na_values=["<NULL>"],
)

# Limpiar espacios y reconocer valores vacíos
for columna in tuni_2025_I.columns:
    tuni_2025_I[columna] = (
        tuni_2025_I[columna]
        .str.strip()
        .replace("", pd.NA)
    )

registros_originales = len(tuni_2025_I)

# Filtrar las entidades incluidas en el análisis
tipos_entidad_incluir = [
    "Universidad",
    "Escuela de Posgrado",
]

tuni_2025_I = tuni_2025_I.loc[
    tuni_2025_I["TIPO_ENTIDAD"].isin(tipos_entidad_incluir)
].copy()

print(f"Registros originales: {registros_originales:,}")
print(f"Registros incluidos: {len(tuni_2025_I):,}")
print(
    f"Registros excluidos: "
    f"{registros_originales - len(tuni_2025_I):,}"
)

display(tuni_2025_I.head())

Registros originales: 1,676,306
Registros incluidos: 1,664,478
Registros excluidos: 11,828


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,PERIODO,PERIODO_ESTANDARIZADO,NIVEL_ACADEMICO,DEPARTAMENTO_LOCAL,GUID_PERSONA
0,260000018,Universidad de Lima,Universidad,Privado,2025-1,2025-1,Maestría,Lima,{1098D4E1-7A34-4B57-ADA6-62D61D4A97FF}
1,260000018,Universidad de Lima,Universidad,Privado,2025-1,2025-1,Maestría,Lima,{8A8DB018-229B-4D26-BEB6-6FCCB07640AD}
2,260000018,Universidad de Lima,Universidad,Privado,2025-1,2025-1,Maestría,Lima,{7369099A-9F7A-4920-9EC0-7B316C3F0633}
3,260000018,Universidad de Lima,Universidad,Privado,2025-1,2025-1,Maestría,Lima,{0BA0410F-8F73-4CFA-9A99-FA4F60B6E33E}
4,260000018,Universidad de Lima,Universidad,Privado,2025-1,2025-1,Maestría,Lima,{1151B198-156D-430A-9B8A-FC93286F2468}


In [37]:
# Clasificar el nivel académico
nivel_academico = (
    tuni_2025_I["NIVEL_ACADEMICO"]
    .str.strip()
    .str.casefold()
)

tuni_2025_I["NIVEL"] = "Posgrado"

tuni_2025_I.loc[
    nivel_academico.eq("carrera profesional").fillna(False),
    "NIVEL",
] = "Pregrado"

tuni_2025_I.loc[
    nivel_academico.isna(),
    "NIVEL",
] = "SIN INFORMACIÓN"

# Limpiar el identificador de persona
tuni_2025_I["GUID_PERSONA"] = (
    tuni_2025_I["GUID_PERSONA"]
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

# Conteos por nivel
tabla_nivel_2025_I = (
    tuni_2025_I.groupby("NIVEL")
    .agg(
        Registros=("GUID_PERSONA", "size"),
        Personas_unicas=("GUID_PERSONA", "nunique"),
        Sin_GUID=("GUID_PERSONA", lambda x: x.isna().sum()),
    )
    .reindex(
        ["Pregrado", "Posgrado",],
        fill_value=0,
    )
)

tabla_nivel_2025_I.index.name = "Nivel"

# Total: contar personas únicas en toda la base filtrada
tabla_nivel_2025_I.loc["TOTAL"] = [
    len(tuni_2025_I),
    tuni_2025_I["GUID_PERSONA"].nunique(),
    tuni_2025_I["GUID_PERSONA"].isna().sum(),
]

display(
    tabla_nivel_2025_I.style.format("{:,.0f}")
)

,Registros,Personas_unicas,Sin_GUID
Nivel,,,
Pregrado,"1,542,358","1,531,079",0
Posgrado,"122,120","120,404",0
TOTAL,"1,664,478","1,649,454",0


In [38]:
# Registros por departamento y nivel: TUNI 2025-I
datos_departamento = tuni_2025_I.assign(
    Departamento=(
        tuni_2025_I["DEPARTAMENTO_LOCAL"]
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
        .fillna("SIN INFORMACIÓN")
    )
)

tabla_departamentos_2025_I = (
    datos_departamento
    .groupby(["Departamento", "NIVEL"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Pregrado", "Posgrado"], fill_value=0)
    .sort_index()
)

tabla_departamentos_2025_I.columns.name = None

# Porcentajes dentro de cada nivel
for nivel in ["Pregrado", "Posgrado"]:
    total = tabla_departamentos_2025_I[nivel].sum()

    tabla_departamentos_2025_I[f"{nivel} (%)"] = (
        tabla_departamentos_2025_I[nivel] / total * 100
        if total > 0 else 0.0
    )

# Ordenar columnas
tabla_departamentos_2025_I = tabla_departamentos_2025_I[
    ["Pregrado", "Pregrado (%)", "Posgrado", "Posgrado (%)"]
]

# Agregar totales
tabla_departamentos_2025_I.loc["TOTAL"] = (
    tabla_departamentos_2025_I.sum()
)

display(
    tabla_departamentos_2025_I.style.format({
        "Pregrado": "{:,.0f}",
        "Pregrado (%)": "{:.2f} %",
        "Posgrado": "{:,.0f}",
        "Posgrado (%)": "{:.2f} %",
    })
)

,Pregrado,Pregrado (%),Posgrado,Posgrado (%)
Departamento,,,,
AMAZONAS,"5,673",0.37 %,480,0.39 %
APURÍMAC,"13,706",0.89 %,542,0.44 %
AREQUIPA,"110,574",7.17 %,"2,327",1.91 %
AYACUCHO,"12,997",0.84 %,0,0.00 %
CAJAMARCA,"30,984",2.01 %,114,0.09 %
CALLAO,"14,215",0.92 %,"3,546",2.90 %
CUSCO,"58,458",3.79 %,"2,893",2.37 %
HUANCAVELICA,"6,672",0.43 %,"3,614",2.96 %
HUÁNUCO,"30,451",1.97 %,"5,471",4.48 %


In [39]:
# Registros y personas únicas por nivel académico detallado
tabla_nivel_detallado_2025_I = (
    tuni_2025_I
    .assign(
        Nivel_academico=(
            tuni_2025_I["NIVEL_ACADEMICO"]
            .str.strip()
            .replace("", pd.NA)
            .fillna("SIN INFORMACIÓN")
        )
    )
    .groupby("Nivel_academico")
    .agg(
        Registros_totales=("GUID_PERSONA", "size"),
        Personas_unicas=("GUID_PERSONA", "nunique"),
        Registros_sin_GUID=("GUID_PERSONA", lambda x: x.isna().sum()),
    )
    .sort_index()
)

tabla_nivel_detallado_2025_I.index.name = "Nivel académico"

# Total de personas únicas en toda la base filtrada
tabla_nivel_detallado_2025_I.loc["TOTAL"] = [
    len(tuni_2025_I),
    tuni_2025_I["GUID_PERSONA"].nunique(),
    tuni_2025_I["GUID_PERSONA"].isna().sum(),
]

display(
    tabla_nivel_detallado_2025_I.style.format("{:,.0f}")
)

,Registros_totales,Personas_unicas,Registros_sin_GUID
Nivel académico,,,
Carrera Profesional,"1,542,358","1,531,079",0
Doctorado,"10,507","10,455",0
Maestría,"88,207","87,630",0
Segunda Especialidad,"23,406","23,264",0
TOTAL,"1,664,478","1,649,454",0


### Tabla de registros 1996, 2010 y 2025

In [40]:
# Registros por nivel académico: 1996, 2010 y 2025-I
niveles = [
    "Pregrado",
    "Segunda especialidad",
    "Maestría",
    "Doctorado",
]

registros_1996 = tabla_nivel_detallado_1996["Registros_totales"].copy()
registros_1996.loc["Pregrado"] = len(pregrado_1996)

registros_2010 = tabla_nivel_detallado_2010["Registros_totales"].copy()
registros_2010.loc["Pregrado"] = len(pregrado_2010)

registros_2025 = (
    tabla_nivel_detallado_2025_I["Registros_totales"]
    .rename(index={
        "Carrera Profesional": "Pregrado",
        "Segunda Especialidad": "Segunda especialidad",
    })
)

tabla_comparativa = pd.DataFrame({
    "1996": registros_1996,
    "2010": registros_2010,
    "2025-I": registros_2025,
}).reindex(niveles)

tabla_comparativa.index.name = "Nivel académico"

display(tabla_comparativa.style.format("{:,.0f}"))

,1996,2010,2025-I
Nivel académico,,,
Pregrado,"339,969","782,970","1,542,358"
Segunda especialidad,"2,597","7,726","23,406"
Maestría,"7,921","44,577","88,207"
Doctorado,300,"4,047","10,507"


In [42]:
# Porcentajes dentro de cada año
for anio in ["1996", "2010", "2025-I"]:
    tabla_comparativa[f"{anio} (%)"] = (
        tabla_comparativa[anio] / tabla_comparativa[anio].sum() * 100
    )

tabla_comparativa = tabla_comparativa[[
    "1996", "1996 (%)",
    "2010", "2010 (%)",
    "2025-I", "2025-I (%)",
]]

display(tabla_comparativa.style.format({
    "1996": "{:,.0f}",
    "1996 (%)": "{:.2f} %",
    "2010": "{:,.0f}",
    "2010 (%)": "{:.2f} %",
    "2025-I": "{:,.0f}",
    "2025-I (%)": "{:.2f} %",
}))

,1996,1996 (%),2010,2010 (%),2025-I,2025-I (%)
Nivel académico,,,,,,
Pregrado,"339,969",96.92 %,"782,970",93.29 %,"1,542,358",92.66 %
Segunda especialidad,"2,597",0.74 %,"7,726",0.92 %,"23,406",1.41 %
Maestría,"7,921",2.26 %,"44,577",5.31 %,"88,207",5.30 %
Doctorado,300,0.09 %,"4,047",0.48 %,"10,507",0.63 %


In [41]:
# Registros por departamento, nivel y año
tabla_comparativa_departamentos = pd.concat({
    "Pregrado": pd.DataFrame({
        "1996": tabla_departamentos_1996["Pregrado"],
        "2010": tabla_departamentos_2010["Pregrado"],
        "2025-I": tabla_departamentos_2025_I["Pregrado"],
    }),
    "Posgrado": pd.DataFrame({
        "1996": tabla_departamentos_1996["Posgrado"],
        "2010": tabla_departamentos_2010["Posgrado"],
        "2025-I": tabla_departamentos_2025_I["Posgrado"],
    }),
}, axis=1).drop(index="TOTAL", errors="ignore").fillna(0).astype(int)

tabla_comparativa_departamentos = (
    tabla_comparativa_departamentos.sort_index()
)
tabla_comparativa_departamentos.index.name = "Departamento"
tabla_comparativa_departamentos.loc["TOTAL"] = (
    tabla_comparativa_departamentos.sum()
)

display(tabla_comparativa_departamentos.style.format("{:,.0f}"))

In [43]:
# Porcentajes por departamento, nivel y año
tabla_porcentajes_departamentos = (
    tabla_comparativa_departamentos
    .div(tabla_comparativa_departamentos.loc["TOTAL"], axis=1)
    .mul(100)
)

display(
    tabla_porcentajes_departamentos.style.format("{:.2f} %")
)